# nb01 — Confounding between batch and diagnosis (Figure 1E) and the strategy map (Figure 2A)

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP9 §9.1 (Figure 1, panel E replaced;
Figure 2, panel A re-rendered). Reviewer 1, comment 1 (confounding) and minor comment 2 (type of
strategy D).

**What this notebook draws**

| Panel | Content | Source table |
|---|---|---|
| Figure 1E (new) | Heat map of sample counts per RNA batch × diagnosis (log scale), single-diagnosis batches marked, side bar with the number of diagnoses per batch (the degree of the batch in the batch–diagnosis graph); bias-corrected Cramér's V and AMI in the title | `output/wp1/bipartite_edges.tsv`, `bridging_batches.tsv`, `confounding_by_strategy.tsv` |
| Figure 2A (re-render) | Circos-Sankey map of the 14 batch removal strategies with plain-word labels; strategy D under the "By biology" arc | `output/wp1/confounding_by_strategy.tsv` (sample count per strategy) |

**Inputs:** the WP1 tables written by `r01_confounding_diagnostics.py` (no S3 access needed).
**Outputs:** `figures_svg/fig1E_confounding.{svg,pdf,png}`, `figures_svg/fig2A_strategy_sankey.{svg,pdf,png}`,
`output/wp9/fig1E_batch_by_diagnosis_counts.tsv` (the plotted matrix).

**Run:** `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 nb01_confounding.ipynb`

Labels follow the plain-word naming rule of plan §1.1: diagnoses and strategies are written in plain
words. RNA batch identifiers (for example `RNASeq_FF_PolyA`) are kept, because the Methods explain them
once and they are the published batch names of Supplementary File 1.

## §0 Configuration
### §0a Imports

In [ ]:
# Standard libraries, plotting and the revision paths (revision_lib is used only for paths and names)
import math
import sys
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib import patheffects
from matplotlib.path import Path as MplPath
import textwrap

REVISION_DIR = Path.cwd()
sys.path.insert(0, str(REVISION_DIR))
import revision_lib as rl  # paths and STRATEGY_DISPLAY_NAMES only (plan WP9)

### §0b Paths and switches

- `WP1_DIR` — tables of `r01_confounding_diagnostics.py`.
- `FIGURE_DIR` — `figures_svg/`; every panel is written as SVG + PDF + PNG (200 dpi).
- `ANNOTATE_COUNTS` (default `False`, decision s8) — print the sample count inside every non-empty heat-map cell.
- `SANKEY_FONT_SIZE` (default 9) — label size of the Circos-Sankey on an 8.5 × 6.5 inch canvas
  cropped to the drawing (the submitted version used 7.5 on a mostly empty 10 × 10 inch canvas).

In [ ]:
# Paths and switches
WP1_DIR = rl.OUTPUT_DIR / "wp1"
FIGURE_DIR = REVISION_DIR / "figures_svg"
TABLE_DIR = rl.OUTPUT_DIR / "wp9"
FIGURE_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATE_COUNTS = False  # small decision s8: 7 pt counts are below the 10 pt rule
SANKEY_FONT_SIZE = 9
GLOBAL_FONT_SIZE = 10

plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 100
plt.rcParams.update(
    {
        "font.size": GLOBAL_FONT_SIZE,
        "axes.titlesize": GLOBAL_FONT_SIZE,
        "axes.labelsize": GLOBAL_FONT_SIZE,
        "xtick.labelsize": GLOBAL_FONT_SIZE,
        "ytick.labelsize": GLOBAL_FONT_SIZE,
        "legend.fontsize": GLOBAL_FONT_SIZE,
    }
)
sns.set_style("ticks")

### §0c Palettes

Copied from `figures_for_article/Introductory_figures_for_article.ipynb` (cell "Canonical project
palettes"; frozen file, read only). Temperature-Split: normal B-cell stages cold (blue / green),
lymphomas warm (red / purple / orange).

In [ ]:
# Article 1 palettes (copied, not imported)
lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

strat_pal = {
    "S0_no_removal": (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad": (1.0, 0.7058, 0.5098),
    "B_extended_bad": (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only": (1.0, 0.6235, 0.6078),
    "D_malignant_only": (0.8156, 0.7333, 1.0),
    "E1_iterative_r1": (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2": (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3": (0.8117, 0.8117, 0.8117),
    "F_microarray_only": (1.0, 0.9960, 0.6392),
    "G_affymetrix_only": (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",
    "J_ff_only": "#7ecbc4",
    "K_ffpe_only": "#f2c27f",
}

### §0d Plain-word diagnosis names and the save helper

The 15 levels of `Diagnosis_cell_type_unified` in the 7,174-sample cohort, in ontogeny order (normal
B-cell stages first, then lymphomas), with the names printed on the figure.

In [ ]:
# Diagnosis level -> plain-word label (ontogeny order: normal stages, then lymphomas)
DIAGNOSIS_PLAIN_NAMES = {
    "Bone_marrow_CD19+": "Bone marrow CD19+ B cells",
    "Immature": "Immature B cells",
    "Naive": "Naive B cells",
    "GC": "Germinal centre B cells",
    "Centroblast": "Centroblasts",
    "Centrocyte": "Centrocytes",
    "B_cells": "B cells (unsorted)",
    "MZ": "Marginal zone B cells",
    "Memory": "Memory B cells",
    "Plasmablast": "Plasmablasts",
    "Plasma": "Plasma cells",
    "Follicular_Lymphoma": "Follicular lymphoma",
    "Diffuse_Large_B_Cell_Lymphoma": "Diffuse large B-cell lymphoma",
    "Burkitt_Lymphoma": "Burkitt lymphoma",
    "High_Grade_B_Cell_Lymphoma": "High-grade B-cell lymphoma",
}


def save_figure(figure: plt.Figure, stem: str) -> None:
    # Write one panel as SVG + PDF + PNG (200 dpi) into figures_svg/
    for extension in ("svg", "pdf", "png"):
        figure.savefig(
            FIGURE_DIR / f"{stem}.{extension}", bbox_inches="tight", dpi=200
        )
    print("saved", stem, "(svg, pdf, png)")

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb01_confounding.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURE_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Load the WP1 tables

- `bipartite_edges.tsv`: one row per (RNA batch, diagnosis) pair with ≥ 1 sample — the edges of the
  batch–diagnosis graph (110 edges expected).
- `bridging_batches.tsv`: one row per batch (29), with the number of diagnoses and the
  connected-component id.
- `confounding_by_strategy.tsv`: one row per strategy (14); row `S0_no_removal` holds the
  whole-cohort statistics (bias-corrected Cramér's V 0.358, AMI 0.263 expected).

In [ ]:
# Read the three WP1 tables
batch_diagnosis_edges = pd.read_csv(WP1_DIR / "bipartite_edges.tsv", sep="\t")
batch_summary = pd.read_csv(WP1_DIR / "bridging_batches.tsv", sep="\t").set_index("RNA_BATCH")
confounding_by_strategy = pd.read_csv(
    WP1_DIR / "confounding_by_strategy.tsv", sep="\t"
).set_index("strategy")

whole_cohort = confounding_by_strategy.loc["S0_no_removal"]
print("edges:", len(batch_diagnosis_edges), "| batches:", len(batch_summary))
print("samples:", int(whole_cohort["n_samples"]))
print("V batch x diagnosis (bias-corrected):", round(whole_cohort["V_batch_x_diagnosis"], 3))
print("AMI batch x diagnosis:", round(whole_cohort["AMI_batch_x_diagnosis"], 3))
print("components of the batch-diagnosis graph:", batch_summary["n_components_in_graph"].iloc[0])
assert int(whole_cohort["n_samples"]) == 7174
assert len(batch_summary) == 29

## §2 Figure 1E — batch × diagnosis heat map

**Layout.** Rows are the 29 RNA batches: multi-diagnosis batches first (sorted by size), then the
single-diagnosis batches (sorted by size), separated by a line. Columns are the 15 diagnoses in
ontogeny order. The cell colour is log10(number of samples); empty cells are white. A colour strip on
the left shows the RNA batch palette of Figure 1A, a strip on top the diagnosis palette of Figure 1B.
The bar chart on the right is the number of diagnoses in each batch (its degree in the batch–diagnosis
graph); single-diagnosis batches have degree 1 and are drawn in grey.

**Expected numbers:** 11 single-diagnosis batches holding 2,068 samples (28.8 %); 18 multi-diagnosis
batches holding 5,106 samples; one connected component.

In [ ]:
# Pivot the edges into a batches x diagnoses count matrix and order rows / columns
count_matrix = batch_diagnosis_edges.pivot_table(
    index="RNA_BATCH",
    columns="Diagnosis_cell_type_unified",
    values="n_samples",
    aggfunc="sum",
    fill_value=0,
)
diagnosis_order = [d for d in DIAGNOSIS_PLAIN_NAMES if d in count_matrix.columns]
assert len(diagnosis_order) == count_matrix.shape[1] == 15

batch_summary["is_single_diagnosis"] = batch_summary["n_diagnoses"] == 1
batch_order = (
    batch_summary.sort_values(["is_single_diagnosis", "n_samples"], ascending=[True, False])
    .index.tolist()
)
count_matrix = count_matrix.loc[batch_order, diagnosis_order]
count_matrix.to_csv(TABLE_DIR / "fig1E_batch_by_diagnosis_counts.tsv", sep="\t")

single_batches = batch_summary.index[batch_summary["is_single_diagnosis"]]
n_single_samples = int(batch_summary.loc[single_batches, "n_samples"].sum())
print("single-diagnosis batches:", len(single_batches), "| samples:", n_single_samples,
      f"({100 * n_single_samples / count_matrix.values.sum():.1f} %)")
assert count_matrix.values.sum() == 7174

In [ ]:
# Draw Figure 1E: batch colour strip | heat map | degree bar chart, with a diagnosis strip on top
n_multi = int((~batch_summary.loc[batch_order, "is_single_diagnosis"]).sum())
log_counts = np.log10(count_matrix.where(count_matrix > 0))

figure = plt.figure(figsize=(9.5, 12))
grid = GridSpec(
    4, 3, figure=figure, width_ratios=[0.35, 15, 3.2],
    height_ratios=[0.35, 29, 7.5, 0.5], wspace=0.03, hspace=0.02,
)
colour_bar_axis = figure.add_subplot(grid[3, 1])
heat_axis = figure.add_subplot(grid[1, 1])
batch_strip_axis = figure.add_subplot(grid[1, 0], sharey=heat_axis)
diagnosis_strip_axis = figure.add_subplot(grid[0, 1], sharex=heat_axis)
degree_axis = figure.add_subplot(grid[1, 2], sharey=heat_axis)

heat_image = heat_axis.imshow(
    log_counts.to_numpy(), aspect="auto", cmap="YlOrRd", vmin=0, vmax=np.log10(1100),
    interpolation="nearest",
)
if ANNOTATE_COUNTS:
    for row_index, batch in enumerate(batch_order):
        for column_index, diagnosis in enumerate(diagnosis_order):
            n_cell = count_matrix.loc[batch, diagnosis]
            if n_cell > 0:
                text_colour = "white" if np.log10(n_cell) > 2.2 else "black"
                heat_axis.text(column_index, row_index, f"{n_cell}", ha="center",
                               va="center", fontsize=7, color=text_colour)
heat_axis.axhline(n_multi - 0.5, color="black", linewidth=1.2)
heat_axis.set_xticks(range(len(diagnosis_order)))
heat_axis.set_xticklabels([DIAGNOSIS_PLAIN_NAMES[d] for d in diagnosis_order],
                          rotation=60, ha="right")
heat_axis.tick_params(axis="y", left=False, labelleft=False)
heat_axis.set_xticks(np.arange(-0.5, len(diagnosis_order)), minor=True)
heat_axis.set_yticks(np.arange(-0.5, len(batch_order)), minor=True)
heat_axis.grid(which="minor", color="#dddddd", linewidth=0.4)
heat_axis.tick_params(which="minor", length=0)

batch_colours = [mcolors.to_rgb(rna_batch_palette.get(b, "#d3d3d3")) for b in batch_order]
batch_strip_axis.imshow(np.array(batch_colours)[:, None, :], aspect="auto")
batch_strip_axis.set_xticks([])
batch_strip_axis.set_yticks(range(len(batch_order)))
batch_strip_axis.set_yticklabels(
    # batch IDs with spaces, as in the legends of Figures 4–5 (consistency review 2026-11-08)
    # plain batch names without n (n is in Figure 1A; Daniil 2026-11-09)
    [batch_display_name(b) for b in batch_order]
)
for label, batch in zip(batch_strip_axis.get_yticklabels(), batch_order):
    if batch_summary.loc[batch, "is_single_diagnosis"]:
        label.set_color("#6e6e6e")
        label.set_fontstyle("italic")
batch_strip_axis.set_ylabel("RNA batch")

diagnosis_colours = [mcolors.to_rgb(lymphoma_ontogeny_palette[d]) for d in diagnosis_order]
diagnosis_strip_axis.imshow(np.array(diagnosis_colours)[None, :, :], aspect="auto")
diagnosis_strip_axis.axis("off")

degree_values = batch_summary.loc[batch_order, "n_diagnoses"].to_numpy()
degree_colours = ["#9e9e9e" if d == 1 else "#3a6ea5" for d in degree_values]
degree_axis.barh(range(len(batch_order)), degree_values, color=degree_colours, height=0.7)
degree_axis.axhline(n_multi - 0.5, color="black", linewidth=1.2)
degree_axis.set_xlabel("Diagnoses\nper batch")
degree_axis.tick_params(axis="y", left=False, labelleft=False)
degree_axis.set_xlim(0, degree_values.max() + 16)  # room for the two group labels
sns.despine(ax=degree_axis, left=True)

colour_bar = figure.colorbar(
    heat_image, cax=colour_bar_axis, orientation="horizontal", ticks=[0, 1, 2, 3]
)
colour_bar.set_ticklabels(["1", "10", "100", "1,000"])
colour_bar.set_label("Samples per batch and diagnosis (log scale)")

degree_axis.text(degree_values.max() + 15.8, n_multi - 0.75, "Multi-diagnosis\nbatches",
                 ha="right", va="bottom", fontsize=9, fontweight="bold")
degree_axis.text(degree_values.max() + 15.8, n_multi - 0.25, "Single-diagnosis\nbatches",
                 ha="right", va="top", fontsize=9, fontweight="bold", color="#6e6e6e")
diagnosis_strip_axis.set_title(
    f"Cramér's V (bias-corrected; batch × {len(diagnosis_order)} diagnoses) = {whole_cohort['V_batch_x_diagnosis']:.3f}; "
    f"adjusted mutual information = {whole_cohort['AMI_batch_x_diagnosis']:.3f}\n"
    f"{len(single_batches)} of {len(batch_order)} batches ({n_single_samples:,} samples, "
    f"{100 * n_single_samples / 7174:.1f} %) hold one diagnosis; "
    f"all batches form one connected batch–diagnosis graph",
    fontsize=GLOBAL_FONT_SIZE, pad=8,
)
save_figure(figure, "fig1E_confounding", slot_mm=(180, None))  # full page width
plt.show()

In [ ]:
# for future - to create a separate 'anti-confounding strategy, with single-diagnosis batches removed'

## §3 Figure 2A — Circos-Sankey map of the 14 batch removal strategies

The drawing code is copied from `Introductory_figures_for_article.ipynb` (cells "Sankey plot of samples
by batch removal strategies"; frozen file, read only). Changes against the submitted version:

1. node labels are the plain-word strategy names (`STRATEGY_DISPLAY_NAMES`, Table 1) instead of the
   internal keys with underscores (plan §1.1);
2. the sample counts are read from `confounding_by_strategy.tsv` (the prepared S3 annotations,
   7,174-sample cohort) instead of being recomputed from the annotation;
3. `matplotlib.path.Path` is imported as `MplPath` (the notebook also uses `pathlib.Path`);
4. layout only: the canvas is cropped to the drawing, labels are wrapped and get a white halo, the
   labels that overlapped in the submitted render are anchored by hand (`LABEL_POSITIONS`), and the
   centre shows the total sample count;
5. the figure is written to `figures_svg/` as SVG + PDF + PNG.

**Minor comment 2 (strategy D).** In the submitted Figure 2A, strategy D (malignant samples only)
already sits under the "By biology" arc; the wrong classification ("By biomaterial") is in Table 1,
which is corrected in the manuscript text (plan WP11). The arcs below are therefore unchanged.

In [ ]:
# Sample count per strategy from the WP1 table (prepared S3 annotations)
counts = confounding_by_strategy["n_samples"].astype(int).to_dict()
for strategy, n_samples in counts.items():
    print(f"{rl.STRATEGY_DISPLAY_NAMES[strategy]:<32} {n_samples:>6,}")
assert counts["S0_no_removal"] == 7174 and len(counts) == 14

In [ ]:
# Geometry helpers (copied from the frozen Introductory_figures_for_article.ipynb)
def transform(x, y, theta_deg):
    # Local ribbon coordinates -> global radial (Cartesian) coordinates
    theta = math.radians(theta_deg)
    X = y * math.cos(theta) - x * math.sin(theta)
    Y = y * math.sin(theta) + x * math.cos(theta)
    return X, Y


def draw_curved_ribbon(ax, y1, theta1, y2, theta2, width, color, R_OUT, alpha=0.65):
    # Cubic Bezier ribbon in radial space; curvature ignores the base hidden under the ring
    d = (y2 - max(y1, R_OUT)) * 0.5
    xL, xR = -width / 2.0, width / 2.0
    p0L, p1L = transform(xL, y1, theta1), transform(xL, max(y1, R_OUT) + d, theta1)
    p2L, p3L = transform(xL, y2 - d, theta2), transform(xL, y2, theta2)
    p0R, p1R = transform(xR, y1, theta1), transform(xR, max(y1, R_OUT) + d, theta1)
    p2R, p3R = transform(xR, y2 - d, theta2), transform(xR, y2, theta2)
    verts = [p0L, p1L, p2L, p3L, p3R, p2R, p1R, p0R, p0L]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.CLOSEPOLY]
    ax.add_patch(patches.PathPatch(MplPath(verts, codes), facecolor=color,
                                   edgecolor="none", alpha=alpha, zorder=2))


def darken_color(color, factor=0.75):
    # Darker version of a palette colour for the strategy boxes
    return tuple(c * factor for c in mcolors.to_rgb(color))


def get_span(w, R):
    # Angular span (degrees) of a ribbon of width w at radius R
    val = max(-1.0, min(1.0, (w / 2.0) / R))
    return math.degrees(math.asin(val)) * 2.0

In [ ]:
# Main drawing function (copied; plain-word labels, wrapped, white halo behind text).
# Hand-placed label anchors (x, y, ha, va in data units) where the automatic placement of the
# submitted code made neighbouring labels overlap; every other label keeps the automatic rule.
LABEL_POSITIONS = {
    "J_ff_only": (20.5, 10.8, "left", "center"),
    "K_ffpe_only": (14.5, 15.0, "left", "center"),
    "F_microarray_only": (-13.5, 13.0, "right", "center"),
    "H_affymetrix_extended": (-18.5, 21.6, "right", "center"),
    "G_affymetrix_only": (-21.5, 29.8, "right", "center"),
    "D_malignant_only": (-20.5, -1.7, "right", "center"),
    "A_confirmed_bad": (17.0, -9.5, "left", "top"),
    "E1_iterative_r1": (27.5, 2.0, "center", "bottom"),
    "E2_iterative_r2": (38.5, -10.5, "center", "top"),
    "E3_iterative_r3": (45.3, -4.0, "center", "bottom"),
}
def create_circos_sankey(figsize=(8.5, 6.5), fontsize=SANKEY_FONT_SIZE, palette=strat_pal):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_aspect("equal")
    ax.axis("off")
    SCALE, R_IN, R_OUT = 0.0013, 5.5, 7.5        # count -> width; inner / outer ring radius
    L1, L2, L3, L4 = 16, 26, 36, 46              # radial levels of the strategy boxes
    RECT_H = 1.0
    nodes = {
        "Ring_J": {"y": R_OUT, "theta": 12}, "Ring_K": {"y": R_OUT, "theta": 48},
        "Ring_C": {"y": R_OUT, "theta": 80}, "Ring_F": {"y": R_OUT, "theta": 124},
        "Ring_D": {"y": R_OUT, "theta": 186}, "Ring_I": {"y": R_OUT, "theta": 258},
        "Ring_A": {"y": R_OUT, "theta": 328},
        "J_ff_only": {"y": L1, "theta": 12}, "K_ffpe_only": {"y": L1, "theta": 48},
        "C_rnaseq_only": {"y": L1, "theta": 80},
        "F_microarray_only": {"y": L1, "theta": 124},
        "D_malignant_only": {"y": L1, "theta": 186},
        "I_rare_batches_removed": {"y": L1, "theta": 258},
        "A_confirmed_bad": {"y": L1, "theta": 328},
        "H_affymetrix_extended": {"y": L2, "theta": 124},
        "G_affymetrix_only": {"y": L3, "theta": 124},
        "B_extended_bad": {"y": L2, "theta": 305},
        "E1_iterative_r1": {"y": L2, "theta": 350},
        "E2_iterative_r2": {"y": L3, "theta": 350},
        "E3_iterative_r3": {"y": L4, "theta": 350},
    }
    arcs = [
        {"label": "By biomaterial",
         "start": 12 - get_span(counts["J_ff_only"] * SCALE, R_OUT) / 2 - 1.5,
         "end": 48 + get_span(counts["K_ffpe_only"] * SCALE, R_OUT) / 2 + 1.5,
         "text_angle": 30},
        {"label": "By platform",
         "start": 80 - get_span(counts["C_rnaseq_only"] * SCALE, R_OUT) / 2 - 1.5,
         "end": 124 + get_span(counts["F_microarray_only"] * SCALE, R_OUT) / 2 + 1.5,
         "text_angle": 95},
        {"label": "By biology",
         "start": 186 - get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2 - 1.5,
         "end": 186 + get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2 + 1.5,
         "text_angle": 186},
        {"label": "Bad batches\nremoval",
         "start": 258 - get_span(counts["I_rare_batches_removed"] * SCALE, R_OUT) / 2 - 1.5,
         "end": 328 + get_span(counts["A_confirmed_bad"] * SCALE, R_OUT) / 2 + 1.5,
         "text_angle": 297},
    ]
    ring_color = palette["S0_no_removal"]
    for arc in arcs:
        ax.add_patch(patches.Wedge((0, 0), R_OUT, arc["start"], arc["end"],
                                   width=(R_OUT - R_IN), facecolor=ring_color,
                                   edgecolor="white", linewidth=1.5, zorder=3))
        th = arc["text_angle"] % 360
        ha, rot = ("right", th - 180) if 90 < th < 270 else ("left", th)
        txt_x, txt_y = transform(0, R_OUT + 0.5, th)
        ax.text(txt_x, txt_y, arc["label"], ha=ha, va="center", rotation=rot,
                fontsize=fontsize, weight="bold", color="#333333", zorder=15,
                path_effects=[patheffects.withStroke(linewidth=3, foreground="white")])
    ax.text(0, 0, f"S0 · No\nremoval\n{counts['S0_no_removal']:,}", ha="center",
            va="center", fontsize=fontsize - 1, color="#111111", zorder=15)
    flows = [
        ("Ring_J", "J_ff_only"), ("Ring_K", "K_ffpe_only"), ("Ring_C", "C_rnaseq_only"),
        ("Ring_F", "F_microarray_only"), ("F_microarray_only", "H_affymetrix_extended"),
        ("H_affymetrix_extended", "G_affymetrix_only"), ("Ring_D", "D_malignant_only"),
        ("Ring_I", "I_rare_batches_removed"), ("Ring_A", "A_confirmed_bad"),
        ("A_confirmed_bad", "B_extended_bad"), ("A_confirmed_bad", "E1_iterative_r1"),
        ("E1_iterative_r1", "E2_iterative_r2"), ("E2_iterative_r2", "E3_iterative_r3"),
    ]
    for parent, child in flows:
        n1, n2 = nodes[parent], nodes[child]
        y1_draw = R_IN if parent.startswith("Ring_") else n1["y"] + RECT_H / 2
        draw_curved_ribbon(ax, y1_draw, n1["theta"], n2["y"] - RECT_H / 2, n2["theta"],
                           width=counts[child] * SCALE, color=palette[child], R_OUT=R_OUT)
    for name, n in nodes.items():
        if name.startswith("Ring_"):
            continue
        w, theta = counts[name] * SCALE, n["theta"]
        corners = [(-w / 2, n["y"] - RECT_H / 2), (w / 2, n["y"] - RECT_H / 2),
                   (w / 2, n["y"] + RECT_H / 2), (-w / 2, n["y"] + RECT_H / 2)]
        ax.add_patch(patches.Polygon([transform(lx, ly, theta) for lx, ly in corners],
                                     facecolor=darken_color(palette[name], 0.70),
                                     edgecolor="none", zorder=4))
        th = theta % 360
        X, Y = transform(0, n["y"], theta)
        dx = (w / 2) * abs(math.sin(math.radians(th)))
        dy = (w / 2) * abs(math.cos(math.radians(th)))
        pad = 0.8
        if name in LABEL_POSITIONS:
            tx, ty, ha, va = LABEL_POSITIONS[name]
        elif 45 <= th <= 135:
            tx, ty, ha, va = X, Y + dy + pad, "center", "bottom"
        elif 225 <= th <= 315:
            tx, ty, ha, va = X, Y - dy - pad, "center", "top"
        elif th < 45 or th > 315:
            tx, ty, ha, va = X + dx + pad, Y, "left", "center"
        else:
            tx, ty, ha, va = X - dx - pad, Y, "right", "center"
        wrapped_name = textwrap.fill(rl.STRATEGY_DISPLAY_NAMES[name], width=16)
        label = f"{wrapped_name}\n(N = {counts[name]:,})"
        ax.text(tx, ty, label, ha=ha, va=va, fontsize=fontsize, color="#111111", zorder=10,
                path_effects=[patheffects.withStroke(linewidth=2.5, foreground="white")])
    ax.set_xlim(-34, 62)   # drawing extent: the E3 box reaches x = 46, G box y = 30
    ax.set_ylim(-34, 42)
    return fig


sankey_figure = create_circos_sankey()
save_figure(sankey_figure, "fig2A_strategy_sankey", slot_mm=(180, None))
plt.show()

## §4 Open issues for Daniil

1. **Figure 2A and minor comment 2.** The submitted Figure 2A already places strategy D under "By
   biology"; only Table 1 says "By biomaterial". The re-render changes the node labels to plain words
   (Table 1 names) and is otherwise identical. Decide whether to swap the panel at all, or keep the
   submitted art and fix Table 1 only. 'Fix table 1 only'
2. **Figure 1E counts printed in the cells** (`ANNOTATE_COUNTS`): useful for the reader, but small
   (7 pt). Switch off if the assembled page is crowded.
3. Figure 1E replaces the old bar plot of sample counts by biology and batches; the old panel D (by
   batches) stays. The Figure 1 legend text is rewritten in plan WP11 (paragraph ¶[87]).